# Model training

In [ ]:
import os
import sys
from pathlib import Path
sys.path.insert(0, os.path.abspath("..")) # for src imports

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
torch.backends.cudnn.benchmark = True

print("PyTorch version:", torch.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("   GPU:", torch.cuda.get_device_name(0))
    print("   CUDA version:", torch.version.cuda)

In [ ]:
NUM_CLASSES = 19
IGNORE_INDEX = 255
DATASET_DIR = "../data/dataset" # train / val / test
MODELS_DIR = "../models" # model save

In [ ]:
VERSION = "v1.0.2"

# SPECS1 =  CPU 10700KF, RAM 32GB, GPU RTX 2080S 8GB VRAM
#Original = (1024, 2048) # 2,097,152
#IMG_SIZE = (512, 1024)  # 524,288
IMG_SIZE = (256, 512)   # 131,072 # OK using SPECS1 (8 batch, 12 workers) 20 epoch = ~23 mins (~6GB VRAM used)
#IMG_SIZE = (128, 256)   # 32,768 # EASY using SPECS1 (8 batch, 12 workers) 20 epoch = ~10 mins (~2GB VRAM used)
#IMG_SIZE = (64, 128)    # 8,192

BATCH_SIZE = 8
NUM_WORKERS = min(12, os.cpu_count() or 1)
print("Dataloader workers:", NUM_WORKERS)

# model
BASE = 32
# optimizer
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
# training
EPOCHS = 20

In [ ]:
VERSION_SIZE = f"{VERSION}-{IMG_SIZE[0]}-{IMG_SIZE[1]}"

MODEL_DIR = Path(MODELS_DIR)/VERSION_SIZE
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL = MODEL_DIR/f"model-{VERSION_SIZE}.pt"
MODEL_CHECKPOINT = MODEL_DIR/f"checkpoint-{VERSION_SIZE}.pt"
MODEL_HISTORY = MODEL_DIR/"history.csv"
MODEL_GIF = MODEL_DIR/"training-gif"

Transforms

In [ ]:
from torchvision import transforms
from src.utils import mask_to_long_tensor

# get transforms for images and masks
image_transform = transforms.Compose([
    # BILINEAR averages neighboring pixels to produce smooth gradients
    transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor() # tensor scaled to [0, 1]
])

label_transform = transforms.Compose([
    # NEAREST preserves original values
    transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(), # tensor without scaling (stays 0-255)
    # remove dummy channel dimension and cast to int64 because PyTorch's CrossEntropyLoss requires this for segmentation. [1, H, W] -> [H, W]
    transforms.Lambda(mask_to_long_tensor) # lambda function here does not work in notebook so moved to utils.py
])

Dataloaders

In [ ]:
import torch
from torch.utils.data import DataLoader
from src.dataset import CityscapesDataset


train_ds = CityscapesDataset(os.path.join(DATASET_DIR, "train"), image_transform, label_transform)
valid_ds = CityscapesDataset(os.path.join(DATASET_DIR, "val"), image_transform, label_transform)
test_ds = CityscapesDataset(os.path.join(DATASET_DIR, "test"), image_transform, label_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True, # useful for stochastic gradient descent
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False, # no need
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)


Create model

In [ ]:
from src.model import UNet
model = UNet(in_channels=3, num_classes=NUM_CLASSES, base=BASE).to(device)

Check formats

In [ ]:
# grab one bactch to inspect shape
# [B = Batch Size, 3 = RGB, H = Height, W = Width]
imgs, masks = next(iter(train_loader))
print("images:", imgs.shape, imgs.dtype)    # should be [B,3,H,W] float32
print("masks :", masks.shape, masks.dtype)  # should be [B,H,W] int64

# run model with no gradient (reduces memory usage) to inspect output shape
# [B = Batch Size, 19 = Classes, H = Height, W = Width]
with torch.no_grad():
    out = model(imgs.to(next(model.parameters()).device))
print("logits:", out.shape)  # should be [B,19,H,W]

In [ ]:
from src.utils import save_prediction, make_gif
fixed_image, _ = valid_loader.dataset[26]

Train model

loop:
- training = forward + backward passes over training set
- validation loss = forward pass over validation set
- evaluate = forward pass over validation set

In [ ]:
import time
import csv
import torch
import torch.nn as nn
from tqdm import tqdm
from src.evaluate import evaluate

criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX) # loss function
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,)

best_valid = float("inf")
history = []

def run_epoch(loader, train: bool):
    model.train(train) # enable/disable dropout
    total_loss = 0.0

    ctx = torch.enable_grad() if train else torch.no_grad() # disable gradient calculation during validation to save memory

    with ctx:
        for imgs, masks in tqdm(loader, leave=False):
            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True).long() # int classes required for loss function

            # Automatically use mixed precision to speed up GPU training and reduce VRAM usage
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(imgs) # logits = raw predictions
                loss = criterion(logits, masks) # calculate how incorrect predictions

            if train:
                optimizer.zero_grad() # clear gradients from previous batch
                scaler.scale(loss).backward() # calculate gradients from loss
                scaler.step(optimizer) # update model parameters using calculated gradients
                scaler.update() # help prevent FP16 underflow

            total_loss += loss.item() # add batch loss to total

    return total_loss / len(loader)

start_time = time.time()
for epoch in range(EPOCHS):
    epoch_start = time.time()

    train = run_epoch(train_loader, train=True) # train = enable dropout
    valid = run_epoch(valid_loader, train=False) # validate = disable dropout

    metrics = evaluate(model, valid_loader, device, num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX)

    save_prediction(model, fixed_image, epoch + 1, MODEL_GIF, device) # save png

    row = {
        "epoch": epoch + 1,
        "train_loss": train,
        "val_loss": valid,
        "pixel_accuracy": metrics["pixel_accuracy"],
        "mean_iou": metrics["mean_iou_class"],
        "mean_class_accuracy": metrics["mean_class_accuracy"],
    }
    history.append(row)

    # save training history
    with open(MODEL_HISTORY, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        writer.writeheader()
        writer.writerows(history)

    # save best model weights
    if valid < best_valid:
        best_valid = valid
        torch.save(model.state_dict(), MODEL)

    # save checkpoint
    torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_valid,
            "history": history,
        },MODEL_CHECKPOINT,
    )

    # calculate estimated time left
    elapsed = time.time() - start_time
    avg_epoch_time = elapsed / (epoch + 1)
    remaining = avg_epoch_time * (EPOCHS - epoch - 1)

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"train loss: {train:.4f} | "
        f"valid loss: {valid:.4f} | "
        f"mIoU: {metrics['mean_iou_class']:.4f} | "
        f"pixel acc: {metrics['pixel_accuracy']:.4f} | "
        f"class acc: {metrics['mean_class_accuracy']:.4f} | "
        f"ETA: {time.strftime('%H:%M:%S', time.gmtime(remaining))}"
    )

print(f"\nBest validation loss: {best_valid:.4f}")
print(f"Model: {MODEL}")
print(f"Checkpoint: {MODEL_CHECKPOINT}")
print(f"History: {MODEL_HISTORY}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(MODEL_HISTORY)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Loss curves
axes[0].plot(df["epoch"], df["train_loss"], label="Train Loss")
axes[0].plot(df["epoch"], df["val_loss"], label="Validation Loss")
axes[0].set_title("Cross-Entropy Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_ylim(0, 2)
axes[0].legend()
axes[0].grid(True)

# Segmentation metrics
axes[1].plot(df["epoch"], df["mean_iou"], label="Mean IoU")
axes[1].plot(df["epoch"], df["pixel_accuracy"], label="Pixel Accuracy")
axes[1].plot(df["epoch"], df["mean_class_accuracy"], label="Mean Class Accuracy")
axes[1].set_title("Validation Metrics")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
make_gif(MODEL_GIF, MODEL_GIF/"training.gif")

Load saved model

In [ ]:
model.load_state_dict(torch.load(MODEL))
model.eval()

Show sample output

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from cityscapesscripts.helpers.labels import labels

# Official colours and names for trainIds
# Source: https://github.com/mcordts/cityscapesScripts/blob/master/cityscapesscripts/helpers/labels.py

# keep only trainIds (0-18), sort by trainId
classes = sorted(
    [l for l in labels if 0 <= l.trainId < 19],
    key=lambda l: l.trainId
)

# matplotlib expects float RGB values [0, 1], so divide by 255 to convert
colors = np.array([l.color for l in classes]) / 255
# create colour map with 19 colours + black for unlabelled
cmap = ListedColormap(colors)
cmap.set_over("black")

# get a test image and prediction
imgs, masks = next(iter(test_loader))
model.eval()
with torch.no_grad():
    pred = model(imgs.to(device)).argmax(1).cpu()

# images
img = imgs[0].permute(1, 2, 0).cpu()
true_mask = masks[0].cpu()
pred_mask = pred[0]

# create plot
fig, ax = plt.subplots(2, 1, figsize=(8, 8))

# matplotlib expects (height, width, 3) instead of (3, height, width) so rearrange
ax[0].imshow(img)
ax[0].imshow(true_mask, cmap=cmap, vmin=0, vmax=18, alpha=0.5)
ax[0].set_title("Image with true mask")

ax[1].imshow(pred_mask, cmap=cmap, vmin=0, vmax=18)
ax[1].set_title("Predicted Mask")

# hide plots axis, numbers, borders
for a in ax:
    a.axis("off")

# colour and name + add black for unlabelled
handles = [Patch(color=colors[i], label=l.name) for i, l in enumerate(classes)]
handles.append(Patch(color="black", label="unlabelled"))

fig.legend(handles=handles, loc="center left", bbox_to_anchor=(0.95, 0.5), title="Classes")

plt.tight_layout(rect=[0, 0, 1, 1])
plt.show()

get Pixel Accuracy, Mean IoU (mIoU), Mean Class Accuracy

In [ ]:
from src.evaluate import evaluate

# evaluate the model on the test set
results = evaluate(model=model, loader=test_loader, device=device, num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX)

# print overall metrics
print(f"Pixel Accuracy:       {results['pixel_accuracy']:.4%}")
print(f"Mean IoU (mIoU):      {results['mean_iou_class']:.4%}")
print(f"Mean Class Accuracy:  {results['mean_class_accuracy']:.4%}")

# print per-class results
print("\nPer-class results:")
print("-" * 55)

for name, class_iou, acc in zip(
    results["class_names"],
    results["iou_per_class"],
    results["accuracy_per_class"]
):
    iou_str = f"{class_iou:.4%}" if np.isfinite(class_iou) else "N/A"
    acc_str = f"{acc:.4%}" if np.isfinite(acc) else "N/A"

    print(
        f"{name:20s} IoU: {iou_str:>10s} | "
        f"Accuracy: {acc_str:>10s}"
    )
